In [ ]:
# Sigla da fonte gravada no frontmatter de cada .md (ex.: "WOAH", "CAPC", "IRIS").
# Troque a cada fonte nova: o build_index.ipynb usa este valor em fontes_rag.
FONTE = "IRIS"

In [ ]:
# Pip Install
!pip install -q docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.9/948.9 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.

In [ ]:
#Import
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import files

In [ ]:
# Opção A: upload direto. Opção B (arquivos grandes): arraste o pdfs.zip para o
# painel de arquivos à esquerda e comente a linha files.upload().

#files.upload()  # selecione o pdfs.zip

extract_dir = Path("pdfs_extraidos")
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile("pdfs.zip") as z:
    z.extractall(extract_dir)

# Descobre automaticamente o nível onde estão "Aquatic Manual" / "Terrestrial Manual"
manuais = [p for p in extract_dir.rglob("*") if p.is_dir() and p.name.endswith("Manual")]
INPUT_DIR = manuais[0].parent if manuais else extract_dir
print("INPUT_DIR =", INPUT_DIR)
print(sum(1 for _ in INPUT_DIR.rglob("*.pdf")), "PDFs encontrados")

INPUT_DIR = pdfs_extraidos
159 PDFs encontrados


In [ ]:
MD_DIR = Path("markdown_output")
MD_DIR.mkdir(exist_ok=True)
Path("json_output").mkdir(exist_ok=True)

for md_src in sorted(INPUT_DIR.rglob("*.md")):
    if md_src.with_suffix(".pdf").exists():
        continue  # o PDF será convertido; ignora o .md antigo
    rel = md_src.relative_to(INPUT_DIR)
    destino = MD_DIR / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(md_src, destino)
        print("COPIADO:", rel)

In [ ]:
%%writefile worker.py
import sys, json, gc
from pathlib import Path
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions
from docling.datamodel.settings import settings
from docling.document_converter import DocumentConverter, PdfFormatOption

MAX_FILES = int(sys.argv[1])
DO_OCR = sys.argv[2] == "1"
input_dir = Path(sys.argv[3])
FONTE = sys.argv[4]

md_dir, json_dir = Path("markdown_output"), Path("json_output")
crashes = Path("crashes.txt")
skip = set(crashes.read_text(encoding="utf-8").splitlines()) if crashes.exists() else set()

def frontmatter(rel):
    p = rel.parts
    manual = p[0] if len(p) > 1 else ""
    cat = p[1] if len(p) > 2 else ""
    return ("---\n"
            f"titulo: {json.dumps(rel.stem, ensure_ascii=False)}\n"
            f"manual: {json.dumps(manual, ensure_ascii=False)}\n"
            f"categoria: {json.dumps(cat, ensure_ascii=False)}\n"
            f"arquivo_origem: {json.dumps(rel.as_posix(), ensure_ascii=False)}\n"
            f"fonte: {json.dumps(FONTE, ensure_ascii=False)}\n---\n\n")

pendentes = []
for pdf in sorted(input_dir.rglob("*.pdf")):
    rel = pdf.relative_to(input_dir)
    if (md_dir / rel).with_suffix(".md").exists() or rel.as_posix() in skip:
        continue
    pendentes.append((pdf, rel))

if not pendentes:
    sys.exit(0)

settings.perf.page_batch_size = 1
opts = PdfPipelineOptions()
opts.do_ocr = DO_OCR
if DO_OCR:
    opts.ocr_options = EasyOcrOptions(lang=["en"])
opts.do_table_structure = True
opts.generate_page_images = False
opts.generate_picture_images = False
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
)

for pdf, rel in pendentes[:MAX_FILES]:
    Path("current.txt").write_text(rel.as_posix(), encoding="utf-8")
    md_path = (md_dir / rel).with_suffix(".md")
    json_path = (json_dir / rel).with_suffix(".json")
    try:
        md_path.parent.mkdir(parents=True, exist_ok=True)
        json_path.parent.mkdir(parents=True, exist_ok=True)
        result = converter.convert(str(pdf))
        doc = result.document
        md_text = frontmatter(rel) + doc.export_to_markdown()
        doc.save_as_json(json_path)
        md_path.write_text(md_text, encoding="utf-8")  # por último: marca como concluído
        n = len(result.pages) or 1
        if len(md_text) / n < 200:
            with open("suspeitos.txt", "a", encoding="utf-8") as f:
                f.write(rel.as_posix() + "\n")
            print("SUSPEITO:", rel, flush=True)
        else:
            print("OK:", rel, flush=True)
        del result, doc, md_text
    except Exception as e:
        with open("erros.txt", "a", encoding="utf-8") as f:
            f.write(f"{rel.as_posix()}: {e}\n")
        print("ERRO:", rel, e, flush=True)
    gc.collect()
    Path("current.txt").unlink(missing_ok=True)

Writing worker.py


In [ ]:
ARQUIVOS_POR_PROCESSO = 8   # se ainda estourar a RAM, reduza para 3-4

def pendentes():
    crashes = Path("crashes.txt")
    skip = set(crashes.read_text(encoding="utf-8").splitlines()) if crashes.exists() else set()
    return [
        p for p in INPUT_DIR.rglob("*.pdf")
        if not (MD_DIR / p.relative_to(INPUT_DIR)).with_suffix(".md").exists()
        and p.relative_to(INPUT_DIR).as_posix() not in skip
    ]

def rodar_tudo(ocr: bool):
    flag = "1" if ocr else "0"
    while pendentes():
        r = subprocess.run([sys.executable, "worker.py", str(ARQUIVOS_POR_PROCESSO), flag, str(INPUT_DIR), FONTE])
        cur = Path("current.txt")
        if cur.exists():  # o worker morreu no meio de um arquivo (provável falta de RAM)
            nome = cur.read_text(encoding="utf-8")
            with open("crashes.txt", "a", encoding="utf-8") as f:
                f.write(nome + "\n")
            cur.unlink()
            print(f"CRASH em {nome} (registrado em crashes.txt)")
        elif r.returncode != 0:
            print("Worker falhou fora de um arquivo. Interrompendo.")
            break
    print("Concluído. Pendentes:", len(pendentes()))

rodar_tudo(ocr=False)

Concluído. Pendentes: 0


In [ ]:
for nome in ["erros.txt", "suspeitos.txt", "crashes.txt"]:
    p = Path(nome)
    linhas = p.read_text(encoding="utf-8").splitlines() if p.exists() else []
    print(f"\n{nome}: {len(linhas)}")
    for l in linhas:
        print("  -", l)

print("\nMarkdowns gerados:", sum(1 for _ in MD_DIR.rglob("*.md")))


erros.txt: 0

suspeitos.txt: 0

crashes.txt: 0

Markdowns gerados: 159


In [ ]:
susp = Path("suspeitos.txt")
if susp.exists():
    lista = sorted(set(susp.read_text(encoding="utf-8").splitlines()))
    susp.unlink()
    for nome in lista:
        (MD_DIR / nome).with_suffix(".md").unlink(missing_ok=True)
        (Path("json_output") / nome).with_suffix(".json").unlink(missing_ok=True)
    print(f"Reprocessando {len(lista)} arquivos com OCR...")
    rodar_tudo(ocr=True)
else:
    print("Nenhum suspeito para reprocessar.")

Nenhum suspeito para reprocessar.


In [ ]:
shutil.make_archive("resultado_md", "zip", ".", "markdown_output")
shutil.make_archive("resultado_json", "zip", ".", "json_output")

files.download("resultado_md.zip")
files.download("resultado_json.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>